# Advanced Evolution Techniques

This notebook demonstrates advanced evolutionary techniques including:
- Novelty Search
- Multi-objective Optimization
- Speciation
- Adaptive Evolution

These techniques help maintain diversity and avoid local optima in evolutionary search.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

from genetic_coil_arena.config import Config
from genetic_coil_arena.evo.arena import EvolutionArena
from genetic_coil_arena.evo.advanced import (
    NoveltySearch, 
    MultiObjectiveEvolution,
    SpeciationEvolution,
    AdaptiveEvolution
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ Advanced evolution toolkit loaded!")

## 1. Novelty Search

Novelty search promotes behavioral diversity by rewarding agents that behave differently from previous solutions.

In [ ]:
# Initialize novelty search
novelty_search = NoveltySearch(k_nearest=10, novelty_threshold=0.1)

# Simulate some behavior characterizations
behaviors = []
novelty_scores = []

# Generate diverse behaviors
for i in range(50):
    # Simulate evaluation results
    eval_results = {
        'avg_score': np.random.exponential(2.0),
        'avg_length': np.random.normal(5, 2),
        'success_rate': np.random.beta(2, 5),
        'avg_steps': np.random.normal(100, 30)
    }
    
    # Create behavior characterization
    behavior = novelty_search.characterize_behavior(None, eval_results)
    behaviors.append(behavior)
    
    # Calculate novelty
    novelty = novelty_search.calculate_novelty(behavior, behaviors[:-1])
    novelty_scores.append(novelty)
    
    # Update archive
    added = novelty_search.update_archive(behavior, novelty)
    if added:
        print(f"Individual {i}: Added to novelty archive (novelty={novelty:.3f})")

# Visualize novelty over time
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(novelty_scores, 'b-', alpha=0.7)
plt.axhline(y=novelty_search.novelty_threshold, color='r', linestyle='--', 
           label=f'Archive threshold ({novelty_search.novelty_threshold})')
plt.title('Novelty Scores Over Time')
plt.xlabel('Individual')
plt.ylabel('Novelty Score')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
archive_sizes = [len(novelty_search.archive) for _ in range(len(novelty_scores))]
plt.plot(archive_sizes, 'g-', linewidth=2)
plt.title('Novelty Archive Growth')
plt.xlabel('Individual')
plt.ylabel('Archive Size')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Novelty Search Results:")
print(f"Final archive size: {len(novelty_search.archive)}")
print(f"Average novelty: {np.mean(novelty_scores):.3f}")
print(f"Novelty standard deviation: {np.std(novelty_scores):.3f}")

## 2. Multi-objective Optimization

Balance fitness maximization with novelty preservation.

In [ ]:
# Initialize multi-objective evolution
mo_evolution = MultiObjectiveEvolution(fitness_weight=0.7, novelty_weight=0.3)

# Simulate population with fitness and novelty scores
population_size = 100
fitness_scores = np.random.exponential(2.0, population_size)
novelty_scores = np.random.gamma(2, 0.5, population_size)

# Normalize scores for comparison
fitness_normalized = (fitness_scores - fitness_scores.min()) / (fitness_scores.max() - fitness_scores.min())
novelty_normalized = (novelty_scores - novelty_scores.min()) / (novelty_scores.max() - novelty_scores.min())

# Calculate multi-objective scores
mo_scores = [mo_evolution.calculate_multi_objective_score(f, n) 
            for f, n in zip(fitness_normalized, novelty_normalized)]

# Calculate Pareto ranks
pareto_ranks = mo_evolution.pareto_ranking(
    list(range(population_size)), fitness_scores, novelty_scores
)

# Visualize Pareto fronts
plt.figure(figsize=(15, 5))

# Pareto front visualization
plt.subplot(1, 3, 1)
colors = plt.cm.viridis(np.array(pareto_ranks) / max(pareto_ranks))
scatter = plt.scatter(fitness_scores, novelty_scores, c=pareto_ranks, 
                     cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='Pareto Rank')
plt.xlabel('Fitness')
plt.ylabel('Novelty')
plt.title('Pareto Fronts')
plt.grid(True, alpha=0.3)

# Multi-objective score distribution
plt.subplot(1, 3, 2)
plt.hist(mo_scores, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Multi-objective Score')
plt.ylabel('Frequency')
plt.title('Multi-objective Score Distribution')
plt.grid(True, alpha=0.3)

# Rank distribution
plt.subplot(1, 3, 3)
rank_counts = np.bincount(pareto_ranks)
plt.bar(range(len(rank_counts)), rank_counts, alpha=0.7, color='lightcoral')
plt.xlabel('Pareto Rank')
plt.ylabel('Count')
plt.title('Pareto Rank Distribution')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analysis
front_1_indices = [i for i, rank in enumerate(pareto_ranks) if rank == 1]
print(f"\n🎯 Multi-objective Analysis:")
print(f"Pareto front 1 size: {len(front_1_indices)} individuals")
print(f"Average fitness in front 1: {np.mean([fitness_scores[i] for i in front_1_indices]):.3f}")
print(f"Average novelty in front 1: {np.mean([novelty_scores[i] for i in front_1_indices]):.3f}")
print(f"Total number of fronts: {max(pareto_ranks)}")

## 3. Speciation Evolution

Maintain diversity through species-based evolution with fitness sharing.

In [ ]:
# Simulate a population for speciation
from genetic_coil_arena.evo.genome import create_random_population

# Create diverse population
population_size = 80
genome_size = 100
population = create_random_population(population_size, genome_size, seed=42)

# Add some fitness values
for i, genome in enumerate(population):
    genome.fitness = np.random.exponential(2.0)

# Initialize speciation
speciation = SpeciationEvolution(compatibility_threshold=0.3, target_species=8)

# Assign species
species_assignments = speciation.assign_species(population)

# Calculate adjusted fitness
adjusted_fitness = speciation.calculate_adjusted_fitness(population)

# Visualize species and fitness
plt.figure(figsize=(15, 10))

# Species size distribution
plt.subplot(2, 3, 1)
species_sizes = [len(members) for members in species_assignments.values()]
plt.bar(range(len(species_sizes)), species_sizes, alpha=0.7, color='lightgreen')
plt.xlabel('Species ID')
plt.ylabel('Species Size')
plt.title('Species Size Distribution')
plt.grid(True, alpha=0.3)

# Fitness comparison
plt.subplot(2, 3, 2)
original_fitness = [genome.fitness for genome in population]
plt.scatter(original_fitness, adjusted_fitness, alpha=0.6)
plt.plot([min(original_fitness), max(original_fitness)], 
         [min(original_fitness), max(original_fitness)], 'r--', alpha=0.5)
plt.xlabel('Original Fitness')
plt.ylabel('Adjusted Fitness (with sharing)')
plt.title('Fitness Sharing Effect')
plt.grid(True, alpha=0.3)

# Species fitness distribution
plt.subplot(2, 3, 3)
species_fitness = {}
for species_id, members in species_assignments.items():
    fitness_values = [member.fitness for member in members]
    species_fitness[species_id] = fitness_values

species_means = [np.mean(fitness_values) for fitness_values in species_fitness.values()]
plt.bar(range(len(species_means)), species_means, alpha=0.7, color='orange')
plt.xlabel('Species ID')
plt.ylabel('Mean Fitness')
plt.title('Species Fitness Comparison')
plt.grid(True, alpha=0.3)

# Compatibility matrix (sample)
plt.subplot(2, 3, 4)
sample_size = min(20, len(population))
sample_genomes = population[:sample_size]
compatibility_matrix = np.zeros((sample_size, sample_size))

for i in range(sample_size):
    for j in range(sample_size):
        if i != j:
            compatibility_matrix[i, j] = speciation.calculate_compatibility(
                sample_genomes[i], sample_genomes[j]
            )

plt.imshow(compatibility_matrix, cmap='viridis', aspect='auto')
plt.colorbar(label='Compatibility Distance')
plt.title('Compatibility Matrix (Sample)')
plt.xlabel('Genome ID')
plt.ylabel('Genome ID')

# Fitness sharing effectiveness
plt.subplot(2, 3, 5)
sharing_effect = np.array(adjusted_fitness) / np.array(original_fitness)
plt.hist(sharing_effect, bins=15, alpha=0.7, color='purple', edgecolor='black')
plt.axvline(x=1.0, color='red', linestyle='--', label='No change')
plt.xlabel('Fitness Ratio (Adjusted/Original)')
plt.ylabel('Frequency')
plt.title('Fitness Sharing Effect Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

# Species diversity over "generations"
plt.subplot(2, 3, 6)
# Simulate species evolution over time
generation_species_count = []
for gen in range(20):
    # Add some noise to population
    for genome in population:
        genome.parameters += np.random.normal(0, 0.05, genome.parameters.shape)
    
    species_assignments = speciation.assign_species(population)
    generation_species_count.append(len(species_assignments))

plt.plot(generation_species_count, 'b-o', linewidth=2)
plt.axhline(y=speciation.target_species, color='r', linestyle='--', 
           label=f'Target ({speciation.target_species})')
plt.xlabel('Generation')
plt.ylabel('Number of Species')
plt.title('Species Count Over Time')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🌱 Speciation Results:")
print(f"Number of species: {len(species_assignments)}")
print(f"Largest species size: {max(species_sizes)}")
print(f"Smallest species size: {min(species_sizes)}")
print(f"Average fitness sharing ratio: {np.mean(sharing_effect):.3f}")
print(f"Diversity index: {np.std(adjusted_fitness) / np.mean(adjusted_fitness):.3f}")

## 4. Adaptive Evolution

Automatically adjust evolution parameters based on performance.

In [ ]:
# Initialize adaptive evolution
adaptive_evo = AdaptiveEvolution(adaptation_interval=5)

# Simulate evolution with adaptation
config = Config()
config.evolution.generations = 50

# Track parameter changes
mutation_sigma_history = [config.evolution.mutation_sigma]
mutation_prob_history = [config.evolution.mutation_prob]
performance_history = []
adaptation_points = []

# Simulate fitness over generations
current_fitness = 1.0
best_fitness_history = [current_fitness]

for generation in range(1, config.evolution.generations):
    # Simulate fitness improvement (with some noise)
    improvement = np.random.normal(0.02, 0.01)  # Base improvement with noise
    
    # Mutation rate affects exploration vs exploitation
    if config.evolution.mutation_sigma > 0.3:
        # High mutation: more exploration, potentially larger improvements
        improvement += np.random.normal(0.01, 0.02)
    else:
        # Low mutation: exploitation, smaller but consistent improvements
        improvement += np.random.normal(0.005, 0.005)
    
    current_fitness += improvement
    best_fitness_history.append(current_fitness)
    
    # Check for adaptation
    if adaptive_evo.should_adapt(generation):
        # Get recent performance
        recent_performance = best_fitness_history[-adaptive_evo.adaptation_interval:]
        
        # Adapt parameters
        old_sigma = config.evolution.mutation_sigma
        old_prob = config.evolution.mutation_prob
        
        config = adaptive_evo.adapt_parameters(config, recent_performance)
        
        adaptation_points.append(generation)
        print(f"Generation {generation}: Adapted mutation σ: {old_sigma:.3f} → {config.evolution.mutation_sigma:.3f}")
        print(f"                      Adapted mutation prob: {old_prob:.3f} → {config.evolution.mutation_prob:.3f}")
    
    mutation_sigma_history.append(config.evolution.mutation_sigma)
    mutation_prob_history.append(config.evolution.mutation_prob)

# Visualize adaptive evolution
plt.figure(figsize=(15, 10))

# Fitness evolution
plt.subplot(2, 3, 1)
plt.plot(best_fitness_history, 'b-', linewidth=2, label='Best Fitness')
for adapt_gen in adaptation_points:
    plt.axvline(x=adapt_gen, color='red', linestyle='--', alpha=0.7)
plt.xlabel('Generation')
plt.ylabel('Best Fitness')
plt.title('Fitness Evolution with Adaptation')
plt.legend()
plt.grid(True, alpha=0.3)

# Mutation sigma adaptation
plt.subplot(2, 3, 2)
plt.plot(mutation_sigma_history, 'g-', linewidth=2)
for adapt_gen in adaptation_points:
    plt.axvline(x=adapt_gen, color='red', linestyle='--', alpha=0.7)
plt.xlabel('Generation')
plt.ylabel('Mutation Sigma')
plt.title('Mutation Rate Adaptation')
plt.grid(True, alpha=0.3)

# Mutation probability adaptation
plt.subplot(2, 3, 3)
plt.plot(mutation_prob_history, 'm-', linewidth=2)
for adapt_gen in adaptation_points:
    plt.axvline(x=adapt_gen, color='red', linestyle='--', alpha=0.7)
plt.xlabel('Generation')
plt.ylabel('Mutation Probability')
plt.title('Mutation Probability Adaptation')
plt.grid(True, alpha=0.3)

# Improvement rate over time
plt.subplot(2, 3, 4)
improvement_rates = np.diff(best_fitness_history)
window_size = 5
smoothed_improvement = np.convolve(improvement_rates, np.ones(window_size)/window_size, mode='valid')
plt.plot(range(window_size-1, len(improvement_rates)), smoothed_improvement, 'orange', linewidth=2)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
for adapt_gen in adaptation_points:
    if adapt_gen < len(smoothed_improvement):
        plt.axvline(x=adapt_gen, color='red', linestyle='--', alpha=0.7)
plt.xlabel('Generation')
plt.ylabel('Improvement Rate (smoothed)')
plt.title('Performance Improvement Rate')
plt.grid(True, alpha=0.3)

# Parameter correlation
plt.subplot(2, 3, 5)
plt.scatter(mutation_sigma_history[1:], improvement_rates, alpha=0.6)
plt.xlabel('Mutation Sigma')
plt.ylabel('Fitness Improvement')
plt.title('Mutation Rate vs Improvement')
plt.grid(True, alpha=0.3)

# Adaptation effectiveness
plt.subplot(2, 3, 6)
pre_adapt_improvements = []
post_adapt_improvements = []

for adapt_gen in adaptation_points:
    if adapt_gen >= 5 and adapt_gen < len(improvement_rates) - 5:
        pre_adapt_improvements.extend(improvement_rates[adapt_gen-5:adapt_gen])
        post_adapt_improvements.extend(improvement_rates[adapt_gen:adapt_gen+5])

if pre_adapt_improvements and post_adapt_improvements:
    plt.boxplot([pre_adapt_improvements, post_adapt_improvements], 
               labels=['Pre-adaptation', 'Post-adaptation'])
    plt.ylabel('Improvement Rate')
    plt.title('Adaptation Effectiveness')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🔄 Adaptive Evolution Results:")
print(f"Total adaptations: {len(adaptation_points)}")
print(f"Initial mutation sigma: {mutation_sigma_history[0]:.3f}")
print(f"Final mutation sigma: {mutation_sigma_history[-1]:.3f}")
print(f"Total fitness improvement: {best_fitness_history[-1] - best_fitness_history[0]:.3f}")
print(f"Average improvement per generation: {np.mean(improvement_rates):.4f}")

if pre_adapt_improvements and post_adapt_improvements:
    print(f"Pre-adaptation improvement: {np.mean(pre_adapt_improvements):.4f} ± {np.std(pre_adapt_improvements):.4f}")
    print(f"Post-adaptation improvement: {np.mean(post_adapt_improvements):.4f} ± {np.std(post_adapt_improvements):.4f}")

## 5. Comparative Analysis

Compare the effectiveness of different advanced techniques.

In [ ]:
# Summary comparison of techniques
techniques = {
    'Standard Evolution': {
        'diversity_maintenance': 2,
        'exploration_capability': 3, 
        'convergence_speed': 4,
        'parameter_sensitivity': 3,
        'computational_cost': 1
    },
    'Novelty Search': {
        'diversity_maintenance': 5,
        'exploration_capability': 5,
        'convergence_speed': 2,
        'parameter_sensitivity': 4,
        'computational_cost': 3
    },
    'Multi-objective': {
        'diversity_maintenance': 4,
        'exploration_capability': 4,
        'convergence_speed': 3,
        'parameter_sensitivity': 3,
        'computational_cost': 3
    },
    'Speciation': {
        'diversity_maintenance': 5,
        'exploration_capability': 4,
        'convergence_speed': 3,
        'parameter_sensitivity': 2,
        'computational_cost': 2
    },
    'Adaptive Evolution': {
        'diversity_maintenance': 3,
        'exploration_capability': 4,
        'convergence_speed': 4,
        'parameter_sensitivity': 1,
        'computational_cost': 2
    }
}

# Create radar plot
from math import pi

categories = list(next(iter(techniques.values())).keys())
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

# Number of variables
N = len(categories)

# Compute angle for each axis
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Plot each technique
colors = plt.cm.Set1(np.linspace(0, 1, len(techniques)))
for i, (technique, scores) in enumerate(techniques.items()):
    values = list(scores.values())
    values += values[:1]  # Complete the circle
    
    ax.plot(angles, values, 'o-', linewidth=2, label=technique, color=colors[i])
    ax.fill(angles, values, alpha=0.25, color=colors[i])

# Add category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels([cat.replace('_', ' ').title() for cat in categories])
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'])
ax.grid(True)

plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.title('Advanced Evolution Techniques Comparison\n(5 = Best, 1 = Worst)', 
         size=16, y=1.08)
plt.tight_layout()
plt.show()

# Summary table
import pandas as pd
df = pd.DataFrame(techniques).T
print("\n📊 Technique Comparison Summary:")
print("=====================================")
print(df.to_string())

print("\n🎯 Recommendations:")
print("- Use Novelty Search when diversity is crucial")
print("- Use Speciation for complex, multi-modal problems")
print("- Use Adaptive Evolution for robust parameter-free optimization")
print("- Use Multi-objective when balancing multiple criteria")
print("- Combine techniques for maximum effectiveness")

## Conclusion

Advanced evolution techniques provide powerful tools for:

1. **Novelty Search**: Maintaining behavioral diversity and avoiding local optima
2. **Multi-objective Optimization**: Balancing competing objectives like fitness and novelty
3. **Speciation**: Natural diversity preservation through reproductive isolation
4. **Adaptive Evolution**: Self-tuning parameters for robust optimization

These techniques can be combined and customized for specific research needs in evolutionary robotics and beyond.